In [21]:
# Import required libraries for data handling, model training, and evaluation

import pandas as pd
import numpy as np
import pickle
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


In [22]:

# 1. Load and merge the datasets to make sure the final dataset contains all required columns
drive.mount('/content/drive')

# Read the main CSV file from Google Drive
df_master = pd.read_csv('/content/drive/MyDrive/master_final_final_final.csv')

# Load the MFCC features stored in the pickle file
with open('/content/drive/MyDrive/mfcc_data.pkl', 'rb') as f:
    mfcc_dict = pickle.load(f)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
# Create a list of column names for the 50 MFCC features
mfcc_vec_cols = [f'mfcc_feat_{i}' for i in range(50)]

# Convert MFCC feature array into a DataFrame with proper column names
mfcc_df = pd.DataFrame(mfcc_dict['mfcc_features'], columns=mfcc_vec_cols)

# Add Video_ID column from filenames to match with the main dataset
mfcc_df['Video_ID'] = mfcc_dict['filename']

# Clean Video_ID in the main dataset by removing ".mp4" and extra spaces
df_master['Video_ID'] = df_master['Video_ID'].astype(str).str.replace('.mp4', '', regex=False).str.strip()

# Clean Video_ID in MFCC dataset the same way to ensure proper matching
mfcc_df['Video_ID'] = mfcc_df['Video_ID'].astype(str).str.replace('.mp4', '', regex=False).str.strip()

# Merge the main dataset with MFCC features using Video_ID
df = pd.merge(df_master, mfcc_df, on='Video_ID', how='inner')

# Remove unnamed or empty columns if they exist
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [24]:
# Display the first 5 rows of the dataset to verify that merging and preprocessing worked correctly

df.head()

,Video_ID,label,similarity,WER,correct_words_#,mfcc_prediction,mfcc_probability,mfcc_svm_decision_score,mfcc_feat_0,mfcc_feat_1,...,mfcc_feat_40,mfcc_feat_41,mfcc_feat_42,mfcc_feat_43,mfcc_feat_44,mfcc_feat_45,mfcc_feat_46,mfcc_feat_47,mfcc_feat_48,mfcc_feat_49
0,0_00000,0,8.146640,100.000000,0,1,0.713175,0.782946,-236.617630,92.429817,...,1789.505629,3464.272444,1735.006472,15.691054,17.774254,21.079627,19.397982,18.500947,19.722032,18.300492
1,0_00001,0,76.973684,42.105263,23,0,0.968595,-3.215874,-407.668823,157.306702,...,2151.846035,4112.542378,2831.515993,17.229187,11.051055,15.226733,16.610441,17.618802,20.712544,30.284682
2,0_00002,0,70.625000,58.823529,15,0,0.811473,-1.412783,-272.981842,84.132591,...,1972.243099,3600.777838,1588.110604,23.540508,16.790347,18.396294,17.768354,17.482005,20.647386,16.886003
3,0_00003,0,74.183976,45.945946,20,1,0.553677,0.140833,-333.608978,99.893410,...,1788.165107,3605.878995,1670.140595,19.345279,14.418756,19.533040,17.153365,17.736857,22.227197,16.922992
4,0_00004,0,58.823529,81.250000,5,1,0.794505,1.190282,-350.967590,128.054321,...,1160.019862,2186.167636,1479.830982,21.573779,15.305312,18.890804,16.583260,18.213436,15.842405,15.238705


In [25]:
# 2. Define the experiments using the same experiment names required in the results table
experiments = {
    # E21 uses MFCC features with WER and similarity scores
    "E21": mfcc_vec_cols + ['WER', 'similarity'],

    # E22 uses MFCC features with WER and the MFCC SVM decision score
    "E22": mfcc_vec_cols + ['WER', 'mfcc_svm_decision_score'],

    # E23 uses only text-based and SVM decision score features
    "E23": ['WER', 'similarity', 'mfcc_svm_decision_score'],

    # E24 uses MFCC prediction output, probability, and SVM decision score
    "E24":['mfcc_prediction','mfcc_probability', 'mfcc_svm_decision_score'],

    # E25 combines MFCC features with all selected additional features
    "E25": mfcc_vec_cols + ['similarity', 'WER', 'correct_words_#',
                            'mfcc_probability', 'mfcc_svm_decision_score','mfcc_prediction']
}

# Create an empty list to store the results of each experiment
results = []

In [26]:
# Loop through each experiment and its selected features
for exp_name, features in experiments.items():

    # Train the model using the scaled training data
    model.fit(X_train_scaled, y_train)

    # 1. Prepare the input features and target label
    X = df[features].copy()
    y = df['label'].astype(int) # Convert labels to integers to avoid warnings

    # 2. Split the data into training, validation, and testing sets
    X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42)

    # 3. Scale the features to improve MLP performance and reduce underfitting
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val.values)   # .values prevents the warning
    X_test_scaled = scaler.transform(X_test.values) # .values prevents the warning

    # 4. Build the MLP model with stronger settings
    model = MLPClassifier(
        hidden_layer_sizes=(256, 128, 64), # Wider hidden layers
        activation='relu',
        solver='adam',
        max_iter=1000,         # Increase the number of training iterations
        early_stopping=True,    # Stop training when the validation score stops improving
        random_state=42
    )

    # Train the MLP model on the scaled training data
    model.fit(X_train_scaled, y_train)

    # 5. Evaluate the model on training and testing data
    train_acc = model.score(X_train_scaled, y_train)
    test_acc = model.score(X_test_scaled, y_test)


    # Store the evaluation results for the current experiment
    results.append({
        "Experiment": exp_name,
        "Train_Acc": round(train_acc, 4),
        "Test_Acc": round(test_acc, 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1-score": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4)
    })

    # Print the training and testing accuracy for the current experiment
    print(f"--- Experiment: {exp_name} ---")
    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Testing Accuracy: {test_acc:.4f}")

    # Check whether the model is overfitting, underfitting, or balanced
    if train_acc - test_acc > 0.10:
        print("⚠️ Warning: Overfitting")
    elif train_acc < 0.70:
        print("⚠️ Warning: Underfitting (Still low, needs more data or tuning)")
    else:
        print("✅ Balanced Performance")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


--- Experiment: E21 ---
Training Accuracy: 0.9853
Testing Accuracy: 0.9441
✅ Balanced Performance


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


--- Experiment: E22 ---
Training Accuracy: 0.9908
Testing Accuracy: 0.9375
✅ Balanced Performance


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


--- Experiment: E23 ---
Training Accuracy: 0.8925
Testing Accuracy: 0.8849
✅ Balanced Performance


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


--- Experiment: E24 ---
Training Accuracy: 0.7654
Testing Accuracy: 0.7587
✅ Balanced Performance


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


--- Experiment: E25 ---
Training Accuracy: 0.9945
Testing Accuracy: 0.9527
✅ Balanced Performance


In [27]:
# --- Display the final results table ---
# Convert the results list into a DataFrame for better visualization
results_df = pd.DataFrame(results)

# Print a title for the results
print("\n--- Final Results Table ---")

# Display the results table in a formatted way
display(results_df)


--- Final Results Table ---


,Experiment,Train_Acc,Test_Acc,Precision,Recall,F1-score,ROC-AUC
0,E21,0.9853,0.9441,0.9515,0.9608,0.9561,0.9879
1,E22,0.9908,0.9375,0.9515,0.9608,0.9561,0.9879
2,E23,0.8925,0.8849,0.9515,0.9608,0.9561,0.9879
3,E24,0.7654,0.7587,0.9515,0.9608,0.9561,0.9879
4,E25,0.9945,0.9527,0.9515,0.9608,0.9561,0.9879
